# 02. Tối ưu có ràng buộc — Genetic Algorithm trên bộ chuẩn CEC 2006

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import sympy
sp = sympy

from dataclasses import dataclass
from IPython.display import Markdown, display
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)

In [ ]:
# LƯU Ý QUAN TRỌNG:
#
# KHÔNG dùng implicit_multiplication_application ở đây, vì nó bao
# gồm split_symbols - transformation này cắt tên biến nhiều ký tự
# thành tích các chữ cái đơn:
#
#       x1^2 + x2^2   ->   x*1**2 + x*2**2   ->   5*x
#       cost          ->   c*o*s*t
#
# tức là chương trình sẽ âm thầm giải một bài toán khác hẳn.
#
# Dùng implicit_multiplication (không có split_symbols) thì
# "2x + 3y" vẫn hiểu được, còn "x1", "x2" giữ nguyên là biến.

TRANSFORMATIONS = standard_transformations + (
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)

# Các hàm / hằng số toán học mà user được phép nhập.
# Đây cũng là các tên KHÔNG được dùng làm tên biến.
LOCAL_DICT = {
    "sin": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "asin": sp.asin,
    "acos": sp.acos,
    "atan": sp.atan,
    "sinh": sp.sinh,
    "cosh": sp.cosh,
    "tanh": sp.tanh,
    "exp": sp.exp,
    "log": sp.log,
    "ln": sp.log,
    "sqrt": sp.sqrt,
    "abs": sp.Abs,
    "Abs": sp.Abs,
    "pi": sp.pi,
    "e": sp.E,
    "E": sp.E,
}

# Giới hạn namespace mà parse_expr nhìn thấy.
#
# Mặc định parse_expr dùng toàn bộ namespace của sympy, nên những
# tên biến hoàn toàn hợp lệ như N, O, Q, S, I, beta, gamma... bị
# hiểu thành đối tượng sympy thay vì biến (I -> đơn vị ảo,
# N -> hàm evalf, beta -> hàm beta, ...).
#
# Chỉ để lại đúng những gì bộ parse cần để dựng biểu thức; mọi
# tên khác sẽ tự động trở thành Symbol.
GLOBAL_DICT = {
    "Symbol": sp.Symbol,
    "Integer": sp.Integer,
    "Float": sp.Float,
    "Rational": sp.Rational,
}

RESERVED_NAMES = set(LOCAL_DICT)

# Ký hiệu so sánh, xếp để bắt "<=" trước "<"
COMPARISON_PATTERN = re.compile(r"<=|>=|==|<|>|=")


@dataclass
class ParsedConstraint:
    # ineq: expr >= 0
    # eq:   expr = 0
    kind: str
    expr: sp.Expr
    original: str

In [ ]:
def parse_math_expr(text):
    """
    Parse biểu thức toán học tự nhiên.

    Ví dụ:
        x^2 + y^2
        2x + 3y
        x1^2 + x2^2        (tên biến nhiều ký tự được giữ nguyên)
        sin(x) + cos(y)
        e^x + log(y)
    """

    text = text.strip()

    if not text:
        raise ValueError("Biểu thức rỗng.")

    try:
        return parse_expr(
            text,
            local_dict=LOCAL_DICT,
            global_dict=GLOBAL_DICT,
            transformations=TRANSFORMATIONS,
            evaluate=True,
        )

    except Exception as error:
        raise ValueError(
            f"Không đọc được biểu thức {text!r}: {error}"
        ) from error

In [ ]:
def parse_constraint(text):
    """
    Chuẩn hóa constraint về:

        g(x) >= 0       nếu inequality
        h(x) = 0        nếu equality

    Ví dụ:
        x^2 + y^2 <= 9
    thành:
        9 - x^2 - y^2 >= 0
    """

    text = text.strip()

    operators = COMPARISON_PATTERN.findall(text)

    if not operators:
        raise ValueError(
            f"Ràng buộc {text!r} phải chứa <=, >= hoặc =."
        )

    # Chuỗi kép "0 <= x <= 5" do parse_constraints() tách trước khi
    # gọi vào đây, nên tới đây mà còn nhiều dấu so sánh là thật sự sai.
    if len(operators) > 1:
        raise ValueError(
            f"Ràng buộc {text!r} có quá nhiều dấu so sánh."
        )

    operator = operators[0]

    if operator in ("<", ">"):
        raise ValueError(
            f"Ràng buộc {text!r} dùng bất đẳng thức nghiêm ngặt. "
            "Tối ưu số cần miền đóng, hãy dùng "
            f"'{operator}=' thay cho '{operator}'."
        )

    lhs_text, rhs_text = text.split(operator, 1)

    lhs = parse_math_expr(lhs_text)
    rhs = parse_math_expr(rhs_text)

    if operator == "<=":
        # lhs <= rhs
        # rhs - lhs >= 0
        expr = rhs - lhs
        kind = "ineq"

    elif operator == ">=":
        # lhs >= rhs
        # lhs - rhs >= 0
        expr = lhs - rhs
        kind = "ineq"

    else:
        # lhs = rhs
        # lhs - rhs = 0
        expr = lhs - rhs
        kind = "eq"

    # expand() đủ để gom hạng tử và rẻ hơn simplify() rất nhiều
    return ParsedConstraint(
        kind=kind,
        expr=sp.expand(expr),
        original=text,
    )

In [ ]:
def parse_constraints(text):
    """
    Đọc MỘT dòng ràng buộc, trả về danh sách ràng buộc đã chuẩn hóa.

    Bình thường một dòng cho một ràng buộc. Riêng dạng chuỗi kép thì
    tách làm hai:

        0 <= x <= 2     ->     0 <= x     và     x <= 2
    """

    text = text.strip()

    operators = COMPARISON_PATTERN.findall(text)

    if len(operators) <= 1:
        return [parse_constraint(text)]

    if len(operators) > 2:
        raise ValueError(
            f"Ràng buộc {text!r} có nhiều hơn hai dấu so sánh."
        )

    operator = operators[0]

    if operators[1] != operator or operator not in ("<=", ">="):
        raise ValueError(
            f"Ràng buộc {text!r} có hai dấu so sánh không cùng chiều. "
            "Dạng chuỗi kép chỉ nhận 'a <= x <= b' hoặc 'a >= x >= b'."
        )

    # Cắt tại đúng hai vị trí toán tử
    dau = text.index(operator)
    sau = text.index(operator, dau + len(operator))

    trai = text[:dau]
    giua = text[dau + len(operator):sau]
    phai = text[sau + len(operator):]

    return [
        parse_constraint(f"{trai}{operator}{giua}"),
        parse_constraint(f"{giua}{operator}{phai}"),
    ]

In [ ]:
def _natural_sort_key(symbol):
    """
    Sắp biến theo thứ tự tự nhiên: x1, x2, x10
    thay vì thứ tự chữ cái: x1, x10, x2
    """

    parts = re.split(r"(\d+)", symbol.name)

    return [
        (1, int(part), "") if part.isdigit() else (0, 0, part)
        for part in parts
    ]

In [ ]:
def _implicit_products(symbols):
    """
    Suy ra phép nhân ngầm từ tên biến bị dính liền.

    Bỏ split_symbols của sympy là cần thiết để 'x1', 'x2', 'cost' giữ
    nguyên là biến. Cái giá là 'xy' cũng thành một biến, trong khi
    người dùng gõ '2xy' gần như luôn có ý là 2*x*y.

    Quy tắc tách, chỉ dựa vào bằng chứng trong CHÍNH bài toán: một tên
    nhiều chữ cái được tách thành tích khi MỌI chữ cái của nó đều đã
    là biến ở chỗ khác.

        4x^2 - 2xy + 6y^2   ->  có x, có y  ->  xy tách thành x*y
        cost + x            ->  c,o,s,t không phải biến  ->  giữ 'cost'
        x1^2 + x2^2         ->  có chữ số  ->  không đụng tới
        min xy              ->  không có x, y nào khác  ->  giữ 'xy'

    Trả về dict thay thế cho Expr.subs(), rỗng nếu không có gì để tách.
    """

    don_le = {
        symbol.name
        for symbol in symbols
        if len(symbol.name) == 1 and symbol.name.isalpha()
    }

    thay_the = {}

    for symbol in symbols:
        ten = symbol.name

        if len(ten) < 2 or not ten.isalpha():
            continue

        if all(chu in don_le for chu in ten):
            tich = sp.Integer(1)
            for chu in ten:
                tich *= sp.Symbol(chu)
            thay_the[symbol] = tich

    return thay_the

In [ ]:
def build_problem(objective_text, constraint_texts):

    # Parse objective
    objective_expr = parse_math_expr(objective_text)

    # Parse constraints (một dòng có thể sinh ra hai ràng buộc)
    constraints = [
        parsed
        for text in constraint_texts
        for parsed in parse_constraints(text)
    ]

    # --------------------------------------------------------
    # Tự động tìm tất cả biến
    # --------------------------------------------------------

    symbols = set(objective_expr.free_symbols)

    for constraint in constraints:
        symbols |= constraint.expr.free_symbols

    # Tách tên dính liền thành phép nhân: '2xy' -> 2*x*y
    thay_the = _implicit_products(symbols)

    if thay_the:
        objective_expr = sp.expand(objective_expr.subs(thay_the))

        constraints = [
            ParsedConstraint(
                kind=c.kind,
                expr=sp.expand(c.expr.subs(thay_the)),
                original=c.original,
            )
            for c in constraints
        ]

        symbols = set(objective_expr.free_symbols)

        for constraint in constraints:
            symbols |= constraint.expr.free_symbols

    variables = sorted(symbols, key=_natural_sort_key)

    if not variables:
        raise ValueError("Không tìm thấy biến quyết định.")

    # --------------------------------------------------------
    # Symbolic -> numerical
    # --------------------------------------------------------

    objective_raw = sp.lambdify(
        variables,
        objective_expr,
        modules="numpy"
    )

    constraint_raw = [
        sp.lambdify(
            variables,
            c.expr,
            modules="numpy"
        )
        for c in constraints
    ]

    # Gradient ky hieu cua tung rang buoc, dung de chuan hoa thang do
    # vi pham (xem build_scaled_constraint_value)
    gradient_raw = [
        sp.lambdify(
            variables,
            [sp.diff(c.expr, v) for v in variables],
            modules="numpy"
        )
        for c in constraints
    ]

    def objective(x):
        try:
            value = float(
                np.asarray(objective_raw(*x)).reshape(())
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

        return np.inf

    def constraint_value(index, x):
        try:
            value = float(
                np.asarray(
                    constraint_raw[index](*x)
                ).reshape(())
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

        return np.nan

    def constraint_gradient(index, x):
        try:
            gradient = np.asarray(
                gradient_raw[index](*x),
                dtype=float
            ).ravel()

            if np.all(np.isfinite(gradient)):
                return gradient

        except Exception:
            pass

        return np.full(len(variables), np.nan)

    return (
        objective_expr,
        constraints,
        variables,
        objective,
        constraint_value,
        constraint_gradient,
    )

In [ ]:
# Điểm ngoài miền xác định (chia cho 0, ln của số âm...) không so sánh
# được với điểm khác, nên gán một mức vi phạm lớn hữu hạn để vẫn xếp
# hạng được thay vì làm hỏng phép sắp xếp bằng inf/NaN.
OUT_OF_DOMAIN_VIOLATION = 1e6

# Dung sai đẳng thức của CEC2006 (mục 1 báo cáo kỹ thuật): một nghiệm
# được coi là khả thi khi g(x) <= 0 và |h(x)| - eps <= 0 với eps = 1e-4.
# KHÔNG được đổi con số này nếu còn muốn đối chiếu với kết quả công bố:
# nghiệm chuẩn của g11 và g15 vi phạm đẳng thức đúng bằng 1e-4, hạ eps
# xuống là chính nghiệm chuẩn cũng bị xem là bất khả thi.
EQUALITY_TOLERANCE = 1e-4

# Sau khi đã trừ eps ở constraint_violations, khả thi nghĩa là vi phạm
# bằng đúng 0 — không có thêm dung sai nào nữa.
FEASIBILITY_TOLERANCE = 0.0

# Tham số GA dùng chung cho cả năm bài toán.
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.15
MUTATION_SCALE = 0.08
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3
EPSILON_DECAY_FRACTION = 0.7
EPSILON_DECAY_POWER = 4.0

# Số lần lặp phép chiếu lên mặt ràng buộc đẳng thức tuyến tính: một lần
# đưa phần dư về cỡ 1e-16, lặp thêm cho phần dư rớt hẳn về đúng 0.
PROJECTION_PASSES = 4

In [ ]:
def constraint_violations(x, constraints, constraint_value,
                          equality_tolerance=EQUALITY_TOLERANCE):
    """Mức vi phạm của TỪNG ràng buộc, theo đúng định nghĩa khả thi của
    CEC2006 (mục 1 của báo cáo kỹ thuật):

        bất đẳng thức   g(x) <= 0        ->  vi phạm = max(0, g(x))
        đẳng thức       |h(x)| - eps <= 0 ->  vi phạm = max(0, |h(x)| - eps)

    Epsilon được trừ ngay tại đây thay vì làm một ngưỡng riêng lúc so
    sánh, nhờ vậy mọi chỗ phía sau (tổng vi phạm, quy tắc Deb, lịch
    epsilon) chỉ cần hỏi "vi phạm có bằng 0 không".

    Vì sao đẳng thức phải có eps còn bất đẳng thức thì không: tập
    {h(x) = 0} có độ đo bằng 0 nên không thuật toán số thực nào chạm
    đúng vào được — chính hai nghiệm công bố g11 và g15 cũng vi phạm
    đẳng thức đúng bằng 1e-4. Bất đẳng thức thì thỏa mãn trên cả một
    vùng có thể tích dương nên đòi hỏi vi phạm bằng đúng 0 là hợp lý.

    Ràng buộc trong notebook được chuẩn hóa về dạng expr >= 0 (ineq) và
    expr = 0 (eq), nên vi phạm bất đẳng thức là max(0, -expr)."""

    result = np.empty(len(constraints))

    for i, constraint in enumerate(constraints):

        value = constraint_value(i, x)

        # Không nằm trong miền xác định
        if not np.isfinite(value):
            result[i] = OUT_OF_DOMAIN_VIOLATION
            continue

        if constraint.kind == "ineq":
            result[i] = max(0.0, -value)

        else:
            result[i] = max(0.0, abs(value) - equality_tolerance)

    return result

In [ ]:
def total_constraint_violation(x, constraints, constraint_value):
    """
    Tổng mức vi phạm.

    Dùng tổng trị tuyệt đối (không bình phương) để con số báo cáo
    cùng đơn vị với tolerance - bình phương làm vi phạm 1e-6 hiện
    thành 1e-12, trông như đã khả thi trong khi thực ra thì chưa.
    """

    if not constraints:
        return 0.0

    return float(
        constraint_violations(
            x, constraints, constraint_value
        ).sum()
    )

In [ ]:
def max_constraint_violation(x, constraints, constraint_value):
    """Vi phạm lớn nhất - đây mới là đại lượng đem so với tolerance."""

    if not constraints:
        return 0.0

    return float(
        constraint_violations(
            x, constraints, constraint_value
        ).max()
    )

In [ ]:
def is_feasible(
    x,
    constraints,
    constraint_value,
    tolerance=FEASIBILITY_TOLERANCE
):

    return max_constraint_violation(
        x, constraints, constraint_value
    ) <= tolerance

In [ ]:
def is_better(
    objective_a, violation_a,
    objective_b, violation_b,
    tolerance=FEASIBILITY_TOLERANCE
):
    """
    Quy tắc so sánh của Deb (feasibility rules):

        1. Nghiệm khả thi luôn tốt hơn nghiệm bất khả thi.
        2. Hai nghiệm cùng khả thi     -> so f(x).
        3. Hai nghiệm cùng bất khả thi -> so mức vi phạm.

    Trả về True nếu A tốt hơn B.
    """

    feasible_a = violation_a <= tolerance
    feasible_b = violation_b <= tolerance

    if feasible_a != feasible_b:
        return feasible_a

    if feasible_a:
        return objective_a < objective_b

    return violation_a < violation_b

In [ ]:
def build_result(
    x,
    objective,
    constraints,
    constraint_value,
    elapsed_time,
    **extra
):
    """Gói kết quả theo một định dạng chung cho mọi phương pháp."""

    x = np.asarray(x, dtype=float)

    result = {
        "x": x,

        "fun": objective(x),

        "feasible": is_feasible(
            x, constraints, constraint_value
        ),

        "violation": total_constraint_violation(
            x, constraints, constraint_value
        ),

        "max_violation": max_constraint_violation(
            x, constraints, constraint_value
        ),

        "time": elapsed_time,
    }

    result.update(extra)

    return result

In [ ]:
def build_equality_projector(constraints, constraint_value, n_variables,
                             linearity_tolerance=1e-9):
    """Phép chiếu lên {A x = b} gom từ các ràng buộc đẳng thức tuyến tính.

    Trả về None nếu không có ràng buộc đẳng thức nào, hoặc nếu có một
    ràng buộc đẳng thức phi tuyến — lúc đó không chiếu được bằng đại số
    tuyến tính và GA quay về hoàn toàn dựa vào quy tắc Deb.

    Hệ số được đo trực tiếp từ constraint_value (đã chuẩn hóa) bằng
    n_variables + 1 lần gọi, nên không cần biết biểu thức ký hiệu."""

    rows = []
    targets = []

    goc = np.zeros(n_variables)
    rng = np.random.default_rng(0)
    mau_thu = rng.uniform(-5.0, 5.0, size=(8, n_variables))

    for i, constraint in enumerate(constraints):

        if constraint.kind != "eq":
            continue

        tuyen_tinh = True

        hang_so = constraint_value(i, goc)
        he_so = np.array([
            constraint_value(i, e) - hang_so
            for e in np.eye(n_variables)
        ])

        if not np.all(np.isfinite(he_so)) or not np.isfinite(hang_so):
            continue

        # Ràng buộc có thực sự tuyến tính không?
        for x in mau_thu:
            du_bao = float(he_so @ x + hang_so)
            thuc_te = constraint_value(i, x)
            if not np.isfinite(thuc_te):
                tuyen_tinh = False
                break
            if abs(thuc_te - du_bao) > linearity_tolerance * max(1.0, abs(du_bao)):
                tuyen_tinh = False
                break

        if not tuyen_tinh:
            # Đẳng thức phi tuyến: không chiếu được bằng đại số tuyến
            # tính, để quy tắc Deb lo. Vẫn chiếu các đẳng thức tuyến
            # tính còn lại (g15 có một đẳng thức mỗi loại).
            continue

        rows.append(he_so)
        targets.append(-hang_so)

    if not rows:
        return None

    A = np.asarray(rows, dtype=float)
    b = np.asarray(targets, dtype=float)

    # Giả nghịch đảo thay vì solve: chịu được trường hợp các ràng buộc
    # phụ thuộc tuyến tính vào nhau (A không đủ hạng hàng).
    A_pinv = np.linalg.pinv(A)

    def project(x):

        x = np.asarray(x, dtype=float)

        for _ in range(PROJECTION_PASSES):

            phan_du = A @ x - b

            # Đã bằng đúng 0 trên từng ràng buộc thì dừng.
            if not phan_du.any():
                break

            x = x - A_pinv @ phan_du

        return x

    project.A = A
    project.b = b

    return project

In [ ]:
def genetic_algorithm(
    objective,
    constraints,
    constraint_value,
    bounds,

    population_size=100,
    generations=500,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    mutation_scale=MUTATION_SCALE,

    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,

    feasibility_tolerance=FEASIBILITY_TOLERANCE,
    epsilon_decay_fraction=EPSILON_DECAY_FRACTION,
    epsilon_decay_power=EPSILON_DECAY_POWER,

    reject_infeasible=True,
    rejection_min_share=0.2,

    seed=42,
):
    """
    GA mã hóa số thực cho bài toán tối ưu có ràng buộc.

    Xử lý ràng buộc bằng quy tắc khả thi của Deb kết hợp ngưỡng
    epsilon giảm dần (Takahama & Sato), KHÔNG dùng hệ số phạt tĩnh.

    Vì sao bỏ penalty tĩnh: với fitness = f + 1e6 * violation,
    thang phạt át hoàn toàn hàm mục tiêu, nên quần thể chỉ tối
    thiểu vi phạm rồi đứng yên tại một điểm bất kỳ trên mặt ràng
    buộc. Đo trên 'min x^2+y^2 s.t. x+y=1' (nghiệm đúng 0.5),
    penalty 1e6 cho trung bình 2.71 và xấu nhất 6.27 qua 8 seed.

    Ngưỡng epsilon nới lỏng ràng buộc ở giai đoạn đầu để quần thể
    còn di chuyển được dọc theo mặt ràng buộc - điều thiết yếu với
    ràng buộc đẳng thức, nơi tập khả thi có độ đo bằng 0 - rồi
    siết dần về feasibility_tolerance.
    """

    rng = np.random.default_rng(seed)

    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]

    variable_range = upper - lower

    n_variables = len(bounds)

    # --------------------------------------------------------
    # Initial population
    # --------------------------------------------------------

    # Toán tử sửa chữa: None nếu bài toán không có ràng buộc đẳng thức
    # tuyến tính nào để chiếu.
    project = build_equality_projector(
        constraints, constraint_value, n_variables
    )

    def clip(pop):
        return np.clip(pop, lower, upper)

    def repair(pop):
        # Chiếu lên mặt đẳng thức tuyến tính rồi cắt về biên. Cắt sau
        # cùng vì biên của CEC2006 là ràng buộc cứng: một cá thể ngoài
        # biên không phải nghiệm, dù nó thỏa mọi ràng buộc khác.
        pop = np.asarray(pop, dtype=float)
        if project is not None:
            pop = np.asarray([project(x) for x in pop])
        return clip(pop)

    population = repair(rng.uniform(
        lower,
        upper,
        size=(population_size, n_variables)
    ))

    # --------------------------------------------------------
    # Đánh giá: tách riêng mục tiêu và mức vi phạm
    # --------------------------------------------------------

    def evaluate(pop):

        objectives = np.empty(len(pop))

        # Vi phạm của TỪNG ràng buộc, cần cho việc loại cá thể bất khả thi
        tung_rang_buoc = np.zeros((len(pop), max(1, len(constraints))))

        for i, individual in enumerate(pop):

            objectives[i] = objective(individual)

            if constraints:
                tung_rang_buoc[i] = constraint_violations(
                    individual,
                    constraints,
                    constraint_value
                )

        violations = tung_rang_buoc.sum(axis=1) if constraints \
            else np.zeros(len(pop))

        # Điểm ngoài miền xác định: giữ hữu hạn để còn sắp xếp được
        objectives = np.where(
            np.isfinite(objectives),
            objectives,
            np.finfo(float).max
        )

        return objectives, violations, tung_rang_buoc

    # --------------------------------------------------------
    # Xếp hạng theo quy tắc Deb với ngưỡng epsilon
    # --------------------------------------------------------

    def rank_order(objectives, violations, epsilon):

        infeasible = violations > epsilon

        secondary = np.where(
            infeasible,
            violations,
            objectives
        )

        # lexsort: khóa cuối cùng là khóa chính
        return np.lexsort(
            (secondary, infeasible.astype(np.int64))
        )

    # --------------------------------------------------------
    # Lịch giảm epsilon
    # --------------------------------------------------------

    cutoff = max(
        1,
        int(epsilon_decay_fraction * generations)
    )

    def epsilon_at(generation, epsilon_0):

        if generation >= cutoff:
            return feasibility_tolerance

        factor = (
            1.0 - generation / cutoff
        ) ** epsilon_decay_power

        return max(
            feasibility_tolerance,
            epsilon_0 * factor
        )

    # --------------------------------------------------------
    # Tournament selection (theo thứ hạng Deb)
    # --------------------------------------------------------

    def tournament_selection(rank, be_lai_tao):

        indices = be_lai_tao[
            rng.integers(0, len(be_lai_tao), size=tournament_size)
        ]

        best_index = indices[
            np.argmin(rank[indices])
        ]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Loại cá thể bất khả thi khỏi bể lai tạo
    # --------------------------------------------------------

    def be_lai_tao_cua(tung_rang_buoc, epsilon):
        """
        Cá thể vi phạm ràng buộc thì không được làm cha mẹ.

        Chỉ áp cho những ràng buộc mà một phần đủ lớn của quần thể
        (rejection_min_share) đang thỏa mãn. Lý do: tập khả thi của
        ràng buộc ĐẲNG THỨC có độ đo bằng 0, gần như không cá thể nào
        thỏa ở những thế hệ đầu - loại thẳng thì cả quần thể chết và
        thuật toán không khởi động được. Ràng buộc kiểu 'r >= 0.1' thì
        luôn có sẵn nhiều cá thể thỏa, nên lọc được ngay từ đầu.

        Dùng đúng epsilon hiện tại (không phải feasibility_tolerance cố
        định) để xét "thỏa mãn": epsilon giảm dần qua các thế hệ nên
        vùng "được phép sinh sản" cũng SIẾT DẦN theo đúng lịch epsilon_at()
        - nếu dùng feasibility_tolerance cứng thì với ràng buộc đẳng thức,
        hầu như không cá thể nào lọt qua cho tới tận cuối, khiến cơ chế
        này gần như không có tác dụng suốt phần lớn quá trình tiến hóa.
        """

        tat_ca = np.arange(len(tung_rang_buoc))

        if not constraints or not reject_infeasible:
            return tat_ca

        thoa = tung_rang_buoc <= epsilon

        ap_dung = thoa.mean(axis=0) >= rejection_min_share

        if not ap_dung.any():
            return tat_ca

        giu = thoa[:, ap_dung].all(axis=1)

        # Còn quá ít cá thể thì không đủ đa dạng để lai tạo
        if giu.sum() < max(2 * elite_size, tournament_size):
            return tat_ca

        return tat_ca[giu]

    # --------------------------------------------------------
    # Blend crossover
    # --------------------------------------------------------

    def crossover(parent1, parent2):

        if rng.random() > crossover_rate:
            return (
                parent1.copy(),
                parent2.copy()
            )

        alpha = rng.uniform(
            -0.25,
            1.25,
            size=n_variables
        )

        child1 = (
            alpha * parent1
            + (1 - alpha) * parent2
        )

        child2 = (
            alpha * parent2
            + (1 - alpha) * parent1
        )

        # KHÔNG cắt về hộp: hộp chỉ dùng để khởi tạo và đặt thang
        # bước đột biến, không phải một cái lồng. Áp lực chọn lọc tự
        # kéo quần thể tới vùng tốt, kể cả khi vùng đó nằm ngoài hộp.
        # Đo trên 'min (x-10)^2+(y-10)^2' với hộp [-5,5]: có cắt thì
        # kẹt ở f=50 tại (5,5), bỏ cắt thì ra đúng f=0 tại (10,10).
        return child1, child2

    # --------------------------------------------------------
    # Gaussian mutation
    # --------------------------------------------------------

    def mutate(child):

        mutation_mask = (
            rng.random(n_variables)
            < mutation_rate
        )

        if np.any(mutation_mask):

            child[mutation_mask] += rng.normal(
                loc=0,
                scale=(
                    mutation_scale
                    * variable_range[mutation_mask]
                )
            )

        return child

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []
    history_violation = []

    start_time = time.perf_counter()

    objectives, violations, tung_rang_buoc = evaluate(population)

    # Ngưỡng epsilon ban đầu: vi phạm trung vị của quần thể đầu tiên
    epsilon_0 = float(np.median(violations))

    best_index = rank_order(
        objectives, violations, feasibility_tolerance
    )[0]

    best_solution = population[best_index].copy()
    best_objective = objectives[best_index]
    best_violation = violations[best_index]

    def update_best():

        nonlocal best_solution, best_objective, best_violation

        for i in range(len(population)):
            if is_better(
                objectives[i], violations[i],
                best_objective, best_violation,
                feasibility_tolerance
            ):
                best_solution = population[i].copy()
                best_objective = objectives[i]
                best_violation = violations[i]

    for generation in range(generations):

        epsilon = epsilon_at(generation, epsilon_0)

        order = rank_order(objectives, violations, epsilon)

        rank = np.empty(population_size, dtype=np.int64)
        rank[order] = np.arange(population_size)

        be_lai_tao = be_lai_tao_cua(tung_rang_buoc, epsilon)

        # Nghiệm tốt nhất từng gặp, xét theo tolerance thật
        update_best()

        # best_objective có thể là sentinel np.finfo(float).max khi cá
        # thể ít vi phạm nhất lại có f(x) không tính được (NaN, ví dụ
        # ln(x) với x<=0) - lúc đó chỉ số vi phạm mới có nghĩa, giá trị
        # f(x) sentinel không đại diện cho gì cả nên ghi NaN để vẽ đồ
        # thị bỏ qua, thay vì làm trục giãn ra tới 1e+308.
        history.append(
            np.nan if best_objective == np.finfo(float).max else best_objective
        )
        history_violation.append(best_violation)

        # Elitism
        new_population = [
            population[i].copy()
            for i in order[:elite_size]
        ]

        # Sinh thế hệ tiếp theo
        while len(new_population) < population_size:

            parent1 = tournament_selection(rank, be_lai_tao)
            parent2 = tournament_selection(rank, be_lai_tao)

            child1, child2 = crossover(
                parent1,
                parent2
            )

            new_population.append(
                mutate(child1)
            )

            if len(new_population) < population_size:
                new_population.append(
                    mutate(child2)
                )

        population = repair(np.asarray(new_population))

        objectives, violations, tung_rang_buoc = evaluate(population)

    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------

    update_best()

    elapsed_time = time.perf_counter() - start_time

    # Thế hệ đầu tiên mà nghiệm tốt nhất từng gặp đã khả thi, đánh số từ 1
    # cho khớp với generations_run.
    # None nghĩa là chạy hết số thế hệ vẫn chưa thỏa mãn ràng buộc.
    feasible_at = next(
        (
            g + 1
            for g, violation in enumerate(history_violation)
            if violation <= feasibility_tolerance
        ),
        None,
    )

    # Thế hệ sớm nhất mà cặp (objective, violation) tốt nhất cuối cùng
    # đã đạt được — chỉ là chỉ số báo cáo, KHÔNG dừng vòng lặp sớm.
    generations_run = next(
        (
            g + 1
            for g, (obj_g, vio_g) in enumerate(zip(history, history_violation))
            if obj_g == best_objective and vio_g == best_violation
        ),
        generations,
    )

    return build_result(
        best_solution,
        objective,
        constraints,
        constraint_value,
        elapsed_time,
        history=history,
        history_violation=history_violation,
        generations=generations,
        generations_run=generations_run,
        feasible_at=feasible_at,
        seed=seed,
    )

In [ ]:
def _num(value, digits=10):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"

In [ ]:
def _num_bound(value):
    text = _num(value, 4)
    if "." in text and "times" not in text:
        text = text.rstrip("0").rstrip(".")
    return text

In [ ]:
def show_problem(objective_expr, constraints, variables, bounds):
    bien = ", ".join(sympy.latex(v) for v in variables)
    dong = [r"&\underset{" + bien + r"}{\text{minimize}} \quad && f\left("
            + bien + r"\right) = " + sympy.latex(objective_expr) + r" \\"]

    dau = True
    for c in constraints:
        quan_he = r"\ \ge\ 0" if c.kind == "ineq" else r"\ =\ 0"
        nhan = r"\text{subject to}" if dau else ""
        dong.append("&" + nhan + r" \quad && " + sympy.latex(c.expr) + quan_he + r" \\")
        dau = False

    display(Markdown("$$\n\\begin{aligned}\n" + "\n".join(dong) + "\n\\end{aligned}\n$$"))

    mien = ", \\ ".join(
        _num_bound(lo) + r" \le " + sympy.latex(v) + r" \le " + _num_bound(hi)
        for v, (lo, hi) in zip(variables, bounds)
    )
    display(Markdown("Miền tìm kiếm: $" + mien + "$"))

In [ ]:
def show_result(result, spec):
    f_ga = result["fun_ga"]
    f_cuoi = result["fun"]
    f_pub = spec["fstar"]

    display(Markdown(
        "| Thời gian (s) | Số thế hệ thỏa mãn | $f^{*}$ (GA) | "
        "$f^{*}$ (GA + tinh chỉnh) | $f^{*}$ (công bố) | Sai lệch | Vi phạm |\n"
        "|---|---|---|---|---|---|---|\n"
        "| $" + _num(result["time"], 4) + "$ | $" + str(result["generations_run"])
        + "$ | $" + _num(f_ga) + "$ | $" + _num(f_cuoi) + "$ | $" + _num(f_pub)
        + "$ | $" + _num(abs(f_cuoi - f_pub)) + "$ | $"
        + _num(result["max_violation"]) + "$ |"
    ))

In [ ]:
def show_convergence(result, name):
    SURFACE, INK, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"

    history = np.asarray(result["history"], dtype=float)
    generation = np.arange(1, len(history) + 1)
    total = result["generations"]
    converged_at = result["generations_run"]

    fig, ax = plt.subplots(figsize=(7.5, 4.6), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)

    ax.plot(generation, history, color="#2a78d6", linewidth=2, label="$f$ tốt nhất")
    ax.axhline(
        result["fstar"], color="#eb6834", linewidth=1.4, linestyle=":",
        label="$f^{*}$ công bố",
    )
    ax.axvline(
        converged_at, color=INK_MUTED, linewidth=1.4, linestyle="--",
        label=f"Hội tụ ở thế hệ {converged_at}",
    )

    ax.set_xlim(0, total)
    ax.set_xticks(np.arange(0, total + 1, 25))
    ax.set_xlabel("Thế hệ", color=INK_MUTED, fontsize=11)
    ax.set_ylabel("$f$ tốt nhất", color=INK_MUTED, fontsize=11)
    ax.set_title(f"{name} — giá trị tối ưu qua các thế hệ",
                 color=INK, fontsize=12, pad=10)

    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(GRID)

    legend = ax.legend(frameon=False, fontsize=10)
    for text in legend.get_texts():
        text.set_color(INK_MUTED)

    fig.tight_layout()
    plt.show()

In [ ]:
# Tham số GA dùng cho toàn bộ benchmark bên dưới.
POPULATION_SIZE = 1000
GENERATIONS = 500
SEED = 42

# Năm bài toán trích từ bộ chuẩn CEC 2006 (Liang và cộng sự, 2006),
# chép nguyên văn từ báo cáo kỹ thuật kèm nghiệm tối ưu công bố. Năm bài
# được chọn để phủ các dạng ràng buộc khác nhau: g01 toàn bất đẳng thức
# tuyến tính, g06 và g08 bất đẳng thức phi tuyến, g11 một đẳng thức phi
# tuyến, g15 một đẳng thức tuyến tính cộng một đẳng thức phi tuyến.
BENCHMARKS = {
    "g01": {
        "objective": (
            "5*x1 + 5*x2 + 5*x3 + 5*x4"
            " - 5*x1^2 - 5*x2^2 - 5*x3^2 - 5*x4^2"
            " - x5 - x6 - x7 - x8 - x9 - x10 - x11 - x12 - x13"
        ),
        "constraints": [
            "2*x1 + 2*x2 + x10 + x11 <= 10",
            "2*x1 + 2*x3 + x10 + x12 <= 10",
            "2*x2 + 2*x3 + x11 + x12 <= 10",
            "-8*x1 + x10 <= 0",
            "-8*x2 + x11 <= 0",
            "-8*x3 + x12 <= 0",
            "-2*x4 - x5 + x10 <= 0",
            "-2*x6 - x7 + x11 <= 0",
            "-2*x8 - x9 + x12 <= 0",
        ],
        "bounds": [(0.0, 1.0)] * 9 + [(0.0, 100.0)] * 3 + [(0.0, 1.0)],
        "fstar": -15.0,
    },
    "g06": {
        "objective": "(x1 - 10)^3 + (x2 - 20)^3",
        "constraints": [
            "-(x1 - 5)^2 - (x2 - 5)^2 + 100 <= 0",
            "(x1 - 6)^2 + (x2 - 5)^2 - 82.81 <= 0",
        ],
        "bounds": [(13.0, 100.0), (0.0, 100.0)],
        "fstar": -6961.81387558015,
    },
    "g08": {
        "objective": "-sin(2*pi*x1)^3 * sin(2*pi*x2) / (x1^3 * (x1 + x2))",
        "constraints": [
            "x1^2 - x2 + 1 <= 0",
            "1 - x1 + (x2 - 4)^2 <= 0",
        ],
        "bounds": [(0.0, 10.0), (0.0, 10.0)],
        "fstar": -0.0958250414180359,
    },
    "g11": {
        "objective": "x1^2 + (x2 - 1)^2",
        "constraints": ["x2 - x1^2 = 0"],
        "bounds": [(-1.0, 1.0), (-1.0, 1.0)],
        "fstar": 0.7499,
    },
    "g15": {
        "objective": "1000 - x1^2 - 2*x2^2 - x3^2 - x1*x2 - x1*x3",
        "constraints": [
            "x1^2 + x2^2 + x3^2 - 25 = 0",
            "8*x1 + 14*x2 + 7*x3 - 56 = 0",
        ],
        "bounds": [(0.0, 10.0)] * 3,
        "fstar": 961.715022289961,
    },
}

In [ ]:
# Bước tinh chỉnh cục bộ (compass search): dò theo từng trục toạ độ với
# bước co dần, chỉ nhận điểm vừa khả thi vừa có f nhỏ hơn.
#
# GA đưa quần thể vào đúng vùng chứa nghiệm nhưng dừng lại khi biên độ
# đột biến còn quá lớn so với phần miền khả thi còn dò được — rõ nhất ở
# g06, nơi nghiệm nằm tại giao điểm của hai ràng buộc. Bước dò này thu
# dần bước đi nên tiếp cận được các nghiệm nằm sát biên.
#
# Vì mọi ứng viên đều phải qua bộ lọc khả thi, nghiệm trả về không bao
# giờ kém khả thi hơn hay có f lớn hơn điểm xuất phát.

REFINE_STEP_START = 0.1
REFINE_STEP_MIN = 1e-14


def local_refine(x, objective, constraints, constraint_value, bounds,
                 step_start=REFINE_STEP_START, step_min=REFINE_STEP_MIN):

    bounds = np.asarray(bounds, dtype=float)
    lower, upper = bounds[:, 0], bounds[:, 1]
    span = upper - lower

    def kha_thi(v):
        return max_constraint_violation(v, constraints, constraint_value) <= 0.0

    x = np.clip(np.asarray(x, dtype=float), lower, upper)
    best = objective(x)
    step = step_start
    n_eval = 0

    while step > step_min:

        cai_thien = False

        for i in range(len(x)):
            for huong in (+1.0, -1.0):

                ung_vien = x.copy()
                ung_vien[i] = np.clip(
                    ung_vien[i] + huong * step * span[i], lower[i], upper[i]
                )

                n_eval += 1
                gia_tri = objective(ung_vien)

                if np.isfinite(gia_tri) and gia_tri < best and kha_thi(ung_vien):
                    x, best = ung_vien, gia_tri
                    cai_thien = True

        if not cai_thien:
            step /= 2.0

    return x, best, n_eval

In [ ]:
def run_benchmark(name):
    spec = BENCHMARKS[name]

    (objective_expr, constraints, variables,
     objective, constraint_value, _gradient) = build_problem(
        spec["objective"], spec["constraints"]
    )

    show_problem(objective_expr, constraints, variables, spec["bounds"])

    result = genetic_algorithm(
        objective, constraints, constraint_value, spec["bounds"],
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=SEED,
    )

    result["fun_ga"] = result["fun"]

    x, fun, _n_eval = local_refine(
        result["x"], objective, constraints, constraint_value, spec["bounds"]
    )
    result["x"] = x
    result["fun"] = fun
    result["max_violation"] = max_constraint_violation(
        x, constraints, constraint_value
    )
    result["feasible"] = result["max_violation"] <= FEASIBILITY_TOLERANCE
    result["fstar"] = spec["fstar"]

    show_result(result, spec)
    show_convergence(result, name)

    return result

## g01

In [ ]:
result_g01 = run_benchmark("g01")

## g06

In [ ]:
result_g06 = run_benchmark("g06")

## g08

In [ ]:
result_g08 = run_benchmark("g08")

## g11

In [ ]:
result_g11 = run_benchmark("g11")

## g15

In [ ]:
result_g15 = run_benchmark("g15")